# Build RAG Index (`faiss.index` + `metadata.parquet`)

Run on a **cluster or serverless** notebook attached to your workspace.
Prerequisites:

- Table `main.scholarships.scheme_corpus` populated (run `scholarship_ingest.ipynb` first).
- **Run the next code cell first** — it `pip install`s dependencies so imports work.
  Edit `REPO_ROOT` to match your clone (Workspace sidebar → right-click repo → **Copy path**).

Do **not** put `%restart_python` in the install cell.

Output: `/Volumes/main/scholarships/rag/faiss.index` and `/Volumes/main/scholarships/rag/metadata.parquet`

In [ ]:
# --- Edit REPO_ROOT, then run this whole cell once before any imports ---
# Copy path from Workspace sidebar (right-click repo folder → Copy path). Examples:
#   /Workspace/Repos/you@domain.com/scholarship-project
#   /Workspace/Users/you@domain.com/scholarship-project
import os
import subprocess
import sys

REPO_ROOT = "/Workspace/Users/<YOUR_EMAIL>/scholarship_schemes_hackathon"  # noqa: E501 — edit me

_src = os.path.join(REPO_ROOT, "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

# Pin faiss-cpu<1.8 + numpy<2 so databricks-connect stays happy.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "numpy>=1.24,<2",
        "pandas>=2.0,<3",
        "faiss-cpu>=1.7.0,<1.8",
        "pyarrow>=14",
        "sentence-transformers>=2.2.0",
    ]
)
print("\u2705 pip: numpy 1.x, pandas<3, faiss-cpu 1.7.x, pyarrow, sentence-transformers")

import faiss
print("\u2705 import faiss OK")

In [ ]:
CATALOG = "main"
SCHEMA = "scholarships"
TABLE = "scheme_corpus"
OUT_DIR = f"/Volumes/{CATALOG}/{SCHEMA}/rag"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

pdf = spark.table(f"{CATALOG}.{SCHEMA}.{TABLE}").select(
    "scheme_id", "scheme_name", "text"
).toPandas()

print(f"Loaded {len(pdf)} rows from {CATALOG}.{SCHEMA}.{TABLE}")
pdf.head(3)

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

print(f"Loading embedding model: {EMBED_MODEL}")
model = SentenceTransformer(EMBED_MODEL)

texts = pdf["text"].fillna("").astype(str).tolist()
print(f"Encoding {len(texts)} texts...")
embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=True)
embeddings = embeddings.astype("float32")
print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
import faiss

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # Inner product = cosine similarity (vectors are L2-normalised above)
index.add(embeddings)
print(f"FAISS index: {index.ntotal} vectors, dim={dim}")

In [ ]:
import os

os.makedirs(OUT_DIR, exist_ok=True)

index_path = os.path.join(OUT_DIR, "faiss.index")
meta_path = os.path.join(OUT_DIR, "metadata.parquet")

faiss.write_index(index, index_path)
print(f"\u2705 FAISS index saved → {index_path}")

# Save only the columns needed for retrieval
meta_df = pdf[["scheme_id", "scheme_name", "text"]].copy().reset_index(drop=True)
meta_df.to_parquet(meta_path, index=False)
print(f"\u2705 Metadata saved → {meta_path} ({len(meta_df)} rows)")

In [ ]:
# Smoke test: reload index and run a sample query
import pandas as pd

test_index = faiss.read_index(index_path)
test_meta = pd.read_parquet(meta_path)

sample_query = "SC student from Maharashtra with annual family income below 1 lakh"
q_emb = model.encode([sample_query], normalize_embeddings=True).astype("float32")
dists, idxs = test_index.search(q_emb, 5)

print(f"Query: {sample_query}")
print("Top 5 results:")
for rank, (dist, idx) in enumerate(zip(dists[0], idxs[0])):
    row = test_meta.iloc[idx]
    print(f"  {rank+1}. [{dist:.4f}] {row['scheme_name']}")

print("\n\u2705 RAG index build complete — ready for Databricks Apps deployment")